In [0]:
SELECT
	a.*, b.*, c.click,
	TRY_CAST(b.PURCHASEPRICE AS DOUBLE) * TRY_CAST(c.click AS INT) AS click_revenue
FROM
	(
	SELECT
		a.sendid,
		lower(a.emailname || '76ers') AS emailname,
		a.emailname AS origemail,
		b.emailaddress,
		b.subscriberkey,
		b.eventdate,
		senttime
	FROM
		kagr_sixers.stage.sfmcsendjobs a
	JOIN kagr_sixers.stage.sfmcsent b
 ON
		a.sendid = b.sendid WHERE a.subject NOT LIKE '%Test%' AND a.subject NOT LIKE '%test%'and a.emailname LIKE '76ers%') A
JOIN 
(
	SELECT
		DISTINCT a.eventname,
		'Single Game' AS ledgername,
		a.insertdate AS saledate,
		'opponent' AS opponent,
		tmsectionname AS sectionname,
		tmrowname AS rowname,
		CONCAT(tmsectionname,tmrowname,firstseat,lastseat) as SEATNUMBER,
		'1' AS seats,
		a.PURCHASEPRICE,
		lower(a.forwardtoemail) AS rawemail
	FROM
		kagr_sixers.STAGE.ARCHTICSTICKETEXCHANGE  a
	WHERE
		a.insertdate IS NOT NULL
		AND purchaseprice > '0' and activityname= 'TE Resale' AND seasonname LIKE '%Regular%' AND seasonname NOT LIKE '%Parking%') b
ON
	a.emailaddress = b.rawemail
LEFT JOIN 
 (
	SELECT
		DISTINCT emailaddress,
		sendid,
		'1' AS click
	FROM
		kagr_njd.stage.sfmcclicks WHERE
	lower(URL) LIKE '%ticket%' OR lower(url) LIKE '%moveableink%' 
	OR lower(url) LIKE '%schedule%' OR lower(url) LIKE '%mi76ers%'
	OR lower(url) LIKE '%ticketmaster%' OR lower(url) LIKE '%fevo%') c
 ON
	a.sendid = c.sendid
	AND a.emailaddress = c.emailaddress
WHERE
	date_trunc('DAY',senttime) < saledate AND dateadd(DAY,3,date_trunc('DAY',senttime))>= saledate AND saledate >= '2021-07-01 00:00:00.000';


/*******************************************************************/